In [4]:
import h5netcdf

fn = "./13901616_nc/wT_sub_12h_blocks.nc"

with h5netcdf.File(fn, "r") as nc:
    print(nc)
    tlen = nc.variables["time"].shape[0]
    vlen = nc.variables["wT_sub_kC"].shape[0]
    print("time length:", nc.variables["time"][-1])
    print("data length:", vlen)
    print("equal?", tlen == vlen)

<h5netcdf.File 'wT_sub_12h_blocks.nc' (mode r)>
Dimensions:
    k: <h5netcdf.Dimension 'k': size 49>
    j: <h5netcdf.Dimension 'j': size 120>
    i: <h5netcdf.Dimension 'i': size 60>
    time: <h5netcdf.Dimension 'time': size 7179 (unlimited)>
Groups:
Variables:
    wT_sub_kC: ('time', 'k', 'j', 'i') float32
    time: ('time',) int64
Attributes:
    hourly_step: 144
    block_size: 12
    time_is_block_start: 1
    box_j: 10
    box_i: 10
    variable: 'wT_sub_kC'
    to_wm2: 1
    origin_iter: 13901616
time length: 26305200
data length: 7179
equal? True


In [1]:
import os
import h5netcdf
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
sys.path.insert(0, str(project_root))

import xarray as xr
import numpy as np

from src.utils import make_grid
from src.utils import build_coarse_grid
from src.utils import read_mitgcm
from src.utils.coarsen_UVWTS import coarse_field
from MITgcmutils import mds
from src.utils import mld_avg
from MITgcmutils import mds


# --------------------- your helper (unchanged idea) ----------------------------------
def heat_flux(
    ds_fine: xr.Dataset,
    W: xr.Dataset,
    T: xr.Dataset,
    boundary_Z: str = "fill",
):
    """
    Compute WT (hourly): WT(t) = W(t) * interp_Zleft(T(t))
    (kept here unchanged for reference; slow if called repeatedly because it rebuilds grid)
    """
    ds = xr.merge([ds_fine, W, T])
    grid_fine = make_grid.make_grid_periodic_x(ds)

    # put T on W-interfaces
    T_w = grid_fine.interp(ds["T"], "Z", to="left", boundary=boundary_Z)  # (k_l,j,i)
    WT = ds["W"] * T_w
    return WT


# --------------------- your submesoscale heat flux (DO NOT MODIFY) -------------------
def submeso_vertical_heat_flux(
    ds_fine: xr.Dataset,
    ds_coarse: xr.Dataset,
    ds_ocean: xr.Dataset,
    ds_ocean_c: xr.Dataset,
    WT: xr.Dataset,
    boundary_Z: str = "fill",
    box_j: int = 10,
    box_i: int = 10,
    to_wm2=True,
):
    """
    
    """
    ds = xr.merge([ds_fine, ds_ocean])
    grid_fine = make_grid.make_grid_periodic_x(ds)
    grid_coarse = make_grid.make_grid_periodic_x(ds_coarse)

    upper = ds["hFacC"]
    lower = ds["hFacC"].shift(k=-1, fill_value=0.0)
    open_frac = xr.ufuncs.minimum(upper, lower).rename({'k': 'k_l'})
    Aface = (open_frac * ds["RAC"]).astype('float32')
    wet_w = Aface > 0

    # put T on W-interfaces (not used later, but you had it)
    T_w = grid_fine.interp(ds["T"], "Z", to="left", boundary=boundary_Z)

    def _area_weighted_boxmean(A: xr.DataArray, WGT: xr.DataArray, box_j=50, box_i=50) -> xr.DataArray:
        num = (A * WGT).coarsen(j=box_j, i=box_i, boundary="trim").sum()
        den = (WGT).coarsen(j=box_j, i=box_i, boundary="trim").sum()
        return xr.where(den > 0, num / den, np.nan)

    def as_ordinal(da: xr.DataArray) -> xr.DataArray:
        da = da.reset_coords(drop=True)
        new_coords = {d: xr.DataArray(np.arange(da.sizes[d]), dims=d) for d in da.dims}
        return da.assign_coords(**new_coords)

    wT_bar_kL = _area_weighted_boxmean(WT.where(wet_w), Aface)
    w_bar_kL = (ds_ocean_c["W"])
    T_bar_kC = (ds_ocean_c["T"])
    T_bar_kL = grid_coarse.interp(T_bar_kC, "Z", to="left", boundary=boundary_Z)
    wT_bar_kL = as_ordinal(wT_bar_kL)

    # meso
    wT_meso_kL = (w_bar_kL * T_bar_kL)
    wT_sub_kL = (wT_bar_kL - wT_meso_kL).rename("wT_sub_kL")
    wT_bar_kL = wT_bar_kL.rename("wT_bar_kL")
    wT_meso_kL = wT_meso_kL.rename("wT_meso_kL")
    w_bar_kL = w_bar_kL.rename("w_bar_kL")
    T_bar_kL = T_bar_kL.rename("T_bar_kL")
    # switch to center
    wT_sub_kC = grid_coarse.interp(wT_sub_kL, "Z", to="center", boundary=boundary_Z).rename("wT_sub_kC")
    wT_meso_kC = grid_coarse.interp(wT_meso_kL, "Z", to="center", boundary=boundary_Z).rename("wT_meso_kC")
    wT_bar_kC = grid_coarse.interp(wT_bar_kL, "Z", to="center", boundary=boundary_Z).rename("wT_bar_kC")
    
    if {'j', 'i'}.issubset(ds_coarse.coords):
        for da in (wT_sub_kL, wT_bar_kL, w_bar_kL, T_bar_kL, wT_sub_kC):
            if 'j' in da.dims:
                da = da.assign_coords(j=ds_coarse['j'])
            if 'i' in da.dims:
                da = da.assign_coords(i=ds_coarse['i'])
        wT_sub_kL, wT_bar_kL, w_bar_kL, T_bar_kL, wT_sub_kC = [x for x in (wT_sub_kL, wT_bar_kL, w_bar_kL, T_bar_kL, wT_sub_kC)]

    if to_wm2:
        rho0, cp = 1035.0, 3994.0
        wT_meso_kC = (rho0 * cp * wT_meso_kC).assign_attrs(units="W m^-2")
        wT_bar_kC = (rho0 * cp * wT_bar_kC).assign_attrs(units="W m^-2")
        wT_sub_kC = (rho0 * cp * wT_sub_kC).assign_attrs(units="W m^-2")
    else:
        for da in (wT_sub_kL, wT_bar_kL, wT_sub_kC):
            da.attrs['units'] = "K m s^-1"
    wT_meso_kC.attrs['long_name'] = "mesoscale vertical heat flux at centers (interp from interfaces)"
    wT_bar_kC.attrs['long_name'] = "total coarse vertical heat flux at centers (interp from interfaces)"
    wT_sub_kC.attrs['long_name'] = "submesoscale vertical heat flux at centers (interp from interfaces)"

    return wT_sub_kC


# --------------------- loader ----------------------------------
def make_mitgcm_loader_onevar(grid_dir, read_mitgcm, var: str, dims):
    def load_snapshot(it: int) -> xr.DataArray:
        da = read_mitgcm._ocean_var(grid_dir, var, it, dims)
        da = da.drop_vars("time", errors="ignore")
        da.name = var
        return da
    return load_snapshot


# --------------------- main worker: compute + save only wT_sub_kC -------------------
def update_12h_submeso_flux_to_netcdf(
    start_time_index: int,
    end_time_index: int,
    load_W,                         # -> DataArray (k_l,j,i)
    load_T,                         # -> DataArray (k,j,i)
    out_nc: str,
    ds_fine: xr.Dataset,
    ds_coarse: xr.Dataset,
    box_j: int = 10,
    box_i: int = 10,
    block_size: int = 12,
    step: int = 144,
    time_name: str = "time",
    boundary_Z: str = "fill",
    to_wm2: bool = True,
    dtype=np.float32,
    skip_missing: bool = False,
    print_every: int = 20,
):
    start_time_index = int(start_time_index)
    end_time_index = int(end_time_index)
    step = int(step)

    if end_time_index < start_time_index:
        raise ValueError("end_time_index must be >= start_time_index")

    os.makedirs(os.path.dirname(out_nc) or ".", exist_ok=True)

    block_span = block_size * step

    saved = set()
    origin_iter = start_time_index

    if os.path.exists(out_nc):
        with h5netcdf.File(out_nc, "r") as nc:
            if time_name in nc.variables:
                saved = set(map(int, nc.variables[time_name][:].tolist()))
            if "origin_iter" in nc.attrs:
                origin_iter = int(nc.attrs["origin_iter"])
            if "box_j" in nc.attrs and int(nc.attrs["box_j"]) != int(box_j):
                raise ValueError("box_j mismatch with existing file")
            if "box_i" in nc.attrs and int(nc.attrs["box_i"]) != int(box_i):
                raise ValueError("box_i mismatch with existing file")

    # n-range for complete blocks
    n0 = int(np.ceil((start_time_index - origin_iter) / block_span)) if start_time_index > origin_iter else 0
    n1 = int(np.floor((end_time_index - origin_iter - (block_size - 1) * step) / block_span))
    if n1 < n0:
        print("Not enough hourly data to form even one full 12-hour block yet.")
        return

    # Build grid_fine ONCE (first time we successfully read W,T)
    grid_fine = None

    def _init_file(example_flux_kC: xr.DataArray):
        spatial_dims = tuple(example_flux_kC.dims)
        spatial_sizes = {d: int(example_flux_kC.sizes[d]) for d in spatial_dims}

        with h5netcdf.File(out_nc, "w") as nc:
            nc.dimensions[time_name] = None
            for d, n in spatial_sizes.items():
                nc.dimensions[d] = n

            nc.attrs["origin_iter"] = int(origin_iter)
            nc.attrs["hourly_step"] = int(step)
            nc.attrs["block_size"] = int(block_size)
            nc.attrs["time_is_block_start"] = 1
            nc.attrs["box_j"] = int(box_j)
            nc.attrs["box_i"] = int(box_i)
            nc.attrs["variable"] = "wT_sub_kC"
            nc.attrs["to_wm2"] = int(bool(to_wm2))

            tvar = nc.create_variable(time_name, (time_name,), dtype="i8", chunks=(1,), maxshape=(None,))
            tvar.attrs["long_name"] = "start index of each 12-hour averaging block"
            tvar.attrs["units"] = "model_iteration_index"

            dims = (time_name,) + spatial_dims
            chunks = (1,) + tuple(spatial_sizes[d] for d in spatial_dims)
            maxshape = (None,) + tuple(spatial_sizes[d] for d in spatial_dims)

            v = nc.create_variable("wT_sub_kC", dims, dtype="f4", chunks=chunks, maxshape=maxshape)
            v.attrs["long_name"] = "submesoscale vertical heat flux at centers (12h block)"
            v.attrs["units"] = "W m^-2" if to_wm2 else "K m s^-1"

    appended = skipped_saved = skipped_missing = 0
    nc_app = None

    try:
        for n in range(n0, n1 + 1):
            s = origin_iter + n * block_span

            if print_every and ((n - n0) % print_every == 0):
                print(f"[try] n={n}/{n1} block_start={s}", flush=True)

            if s in saved:
                skipped_saved += 1
                continue

            block_iters = [s + m * step for m in range(block_size)]

            W0 = T0 = None
            sumW = sumT = sumWT = None

            try:
                for it in block_iters:
                    W = load_W(int(it)).astype(dtype)
                    T = load_T(int(it)).astype(dtype)

                    if W0 is None:
                        W0, T0 = W, T
                        sumW = W.values.astype(dtype, copy=True)
                        sumT = T.values.astype(dtype, copy=True)
                        sumWT = np.zeros_like(sumW, dtype=dtype)

                        # build grid_fine ONCE for the whole run
                        if grid_fine is None:
                            ds_tmp = xr.merge([ds_fine, xr.Dataset({"W": W0, "T": T0})])
                            grid_fine = make_grid.make_grid_periodic_x(ds_tmp)
                    else:
                        sumW += W.values
                        sumT += T.values

                    # CORRECT: hourly WT = W(t) * interp_Zleft(T(t))
                    T_w = grid_fine.interp(T, "Z", to="left", boundary=boundary_Z)  # (k_l,j,i)
                    sumWT += (W.values * T_w.values).astype(dtype, copy=False)

            except Exception as e:
                if skip_missing:
                    skipped_missing += 1
                    print(f"  -> missing/unreadable snapshot in block_start={s}, skip ({e})", flush=True)
                    continue
                raise

            # 12h means
            W_12mean = W0.copy(data=(sumW / block_size)).astype("float32") # keep the coordinate as the W0
            T_12mean = T0.copy(data=(sumT / block_size)).astype("float32")
            WT_12mean = W0.copy(data=(sumWT / block_size)).astype("float32")  # (k_l,j,i)

            # build ds_ocean and ds_ocean_c (as you described)
            ds_ocean = xr.Dataset(dict(W=W_12mean, T=T_12mean))
            W_coarse = coarse_field("W", W_12mean, ds_fine, box_j=box_j, box_i=box_i)
            T_coarse = coarse_field("T", T_12mean, ds_fine, box_j=box_j, box_i=box_i)
            ds_ocean_c = xr.Dataset(dict(W=W_coarse, T=T_coarse))

            # compute submesoscale flux for this block, save only this
            wT_sub_kC = submeso_vertical_heat_flux(
                ds_fine=ds_fine,
                ds_coarse=ds_coarse,
                ds_ocean=ds_ocean,
                ds_ocean_c=ds_ocean_c,
                WT=WT_12mean,
                boundary_Z=boundary_Z,
                box_j=box_j,
                box_i=box_i,
                to_wm2=to_wm2,
            ).astype("float32")

            # init file on first successful output
            if not os.path.exists(out_nc):
                _init_file(wT_sub_kC)

            if nc_app is None:
                nc_app = h5netcdf.File(out_nc, "a")

            tvar = nc_app.variables[time_name]
            write_idx = int(tvar.shape[0])

            v = nc_app.variables["wT_sub_kC"]
            spatial_shape = v.shape[1:]

            v._h5ds.resize((write_idx + 1,) + spatial_shape)
            v[write_idx, ...] = wT_sub_kC.values.astype(np.float32, copy=False)

            tvar._h5ds.resize((write_idx + 1,))
            tvar[write_idx] = s

            saved.add(s)
            appended += 1

    finally:
        if nc_app is not None:
            nc_app.close()

    print(
        f"\nUpdated NetCDF (saved only submeso flux): {out_nc}\n"
        f"  appended: {appended}\n"
        f"  skipped (already saved): {skipped_saved}\n"
        f"  skipped (missing hours): {skipped_missing}\n"
        f"  origin_iter={origin_iter}, step={step}, block_size={block_size}\n"
        f"  coarse box: box_j={box_j}, box_i={box_i}",
        flush=True,
    )


# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    path_scratch= "/scratch/jczhang/"
    path_MITgcm= "MITgcm_1km/verification/tutorial_reentrant_channel/output"
    grid_dir = path_scratch + path_MITgcm

    ds_fine = read_mitgcm.read_mitgrid_staggered(grid_dir)

    box_j, box_i = 50, 50
    ds_coarse = build_coarse_grid.build_coarse_grid_staggered(ds_fine, box_j=box_j, box_i=box_i)

    dims_map = {
        "W": ("k_l", "j", "i"),
        "T": ("k", "j", "i"),
    }

    nStep1, nStep2, dS = 1200060 + 180 + 1980, 1279800 + 1, 180

    load_W = make_mitgcm_loader_onevar(grid_dir, read_mitgcm, var="W", dims=dims_map["W"])
    load_T = make_mitgcm_loader_onevar(grid_dir, read_mitgcm, var="T", dims=dims_map["T"])

    out_nc = "./wT_sub_12h_blocks.nc"

    update_12h_submeso_flux_to_netcdf(
        start_time_index=nStep1,
        end_time_index=nStep2,
        load_W=load_W,
        load_T=load_T,
        out_nc=out_nc,
        ds_fine=ds_fine,
        ds_coarse=ds_coarse,
        box_j=box_j,
        box_i=box_i,
        block_size=6,
        step=dS,
        time_name="time",
        skip_missing=False,
        print_every=1,
    )

[try] n=0/71 block_start=1202220
[try] n=1/71 block_start=1203300
[try] n=2/71 block_start=1204380
[try] n=3/71 block_start=1205460
[try] n=4/71 block_start=1206540
[try] n=5/71 block_start=1207620
[try] n=6/71 block_start=1208700
[try] n=7/71 block_start=1209780
[try] n=8/71 block_start=1210860
[try] n=9/71 block_start=1211940
[try] n=10/71 block_start=1213020
[try] n=11/71 block_start=1214100
[try] n=12/71 block_start=1215180
[try] n=13/71 block_start=1216260
[try] n=14/71 block_start=1217340
[try] n=15/71 block_start=1218420
[try] n=16/71 block_start=1219500
[try] n=17/71 block_start=1220580
[try] n=18/71 block_start=1221660
[try] n=19/71 block_start=1222740
[try] n=20/71 block_start=1223820
[try] n=21/71 block_start=1224900
[try] n=22/71 block_start=1225980
[try] n=23/71 block_start=1227060
[try] n=24/71 block_start=1228140
[try] n=25/71 block_start=1229220
[try] n=26/71 block_start=1230300
[try] n=27/71 block_start=1231380
[try] n=28/71 block_start=1232460
[try] n=29/71 block_star